In [ ]:
from tensorflow.keras.applications  import MobileNetV2 , ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tools.attentionBlocks import CBAM
import tools.model_utility as mu
import tensorflow as tf
import keras
from keras.callbacks import ModelCheckpoint

In [ ]:
# data loading and augmentation 
#--------------
aug_data_train = ImageDataGenerator(
    rescale=1./255 ,
    rotation_range=20,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest',
)
aug_data_val = ImageDataGenerator(
    rescale=1./255 ,
)
aug_data_test = ImageDataGenerator(
    rescale=1./255 ,
)
#---------------------
train_img = aug_data_train.flow_from_directory(
    'data/train' ,
    target_size=(224, 224),
    class_mode='binary',
    batch_size=1000,
    shuffle=True,
)
#-----------------------
val_img = aug_data_val.flow_from_directory(
    'data/val' ,
    target_size=(224, 224),
    class_mode='binary',
    batch_size=16,
)
#-----------------------
test_img = aug_data_test.flow_from_directory(
    'data/test' ,
    target_size=(224, 224),
    class_mode='binary',
    batch_size=1000,
)

In [ ]:
# inserting attention blocks to our model

base_model  = ResNet50(input_shape=(224 , 224 , 3) , weights='imagenet' , include_top=False)

def insert_cbam(layer, *args, **kwargs):
    out = layer(*args, **kwargs)
    #add_id = layer.name.split("_" , 2)[-1]
    if layer.name == "conv5_block3_3_bn":
        out = CBAM(out)
    return out
  
resnet_cbam = mu.insert_layer(
    base_model , 
    insert_cbam
)

resnet_cbam.summary()

In [4]:
# unfreeze weights after classifier optimized
for layer in resnet_cbam.layers[1:] :
  if layer.name == "conv5_block3_3_conv" :
    layer.trainable = True
  if layer.name == "conv5_block3_2_conv" :
    layer.trainable = True
  if layer.name == "conv5_block3_1_conv" :
    layer.trainable = True
  if layer.name == "conv5_block2_3_conv" :
    layer.trainable = True
  if layer.name == "conv5_block2_2_conv" :
    layer.trainable = True
  if layer.name == "conv5_block2_1_conv" :
    layer.trainable = True
  if layer.name == "conv5_block1_3_conv" :
    layer.trainable = True
  if layer.name == "conv5_block1_2_conv" :
    layer.trainable = True
  if layer.name == "conv5_block1_1_conv" :
    layer.trainable = True

In [ ]:
# trainig model

check_final_val = ModelCheckpoint(
    'Drive/MyDrive/breast-calcification/models_final/m_loss.keras',
    monitor='val_loss' ,
    save_best_only=True ,
    mode='min'
)
check_final_auc = ModelCheckpoint(
    'Drive/MyDrive/breast-calcification/models_final/m_auc.keras',
    monitor='val_auc' ,
    save_best_only=True ,
    mode='max'
)
csv_log = keras.callbacks.CSVLogger("Drive/MyDrive/breast-calcification/tr_logs/m.csv")
#------
when = [149*30]
lr   = [1e-5 , 1e-6]
Lrschedule = keras.optimizers.schedules.PiecewiseConstantDecay(
    when ,
    lr ,
)
#------
resnet_cbam.compile(
    optimizer = keras.optimizers.Adam(learning_rate=Lrschedule) ,
    loss      = keras.losses.BinaryCrossentropy() ,
    metrics   = [
        "accuracy",
        keras.metrics.AUC(),
        keras.metrics.Recall()
    ]
)
#------
proc = resnet_cbam.fit(
        x=train_img ,
        validation_data=val_img,
        epochs=70,
        callbacks=[check_final_val , check_final_auc , csv_log]
    )
